In [5]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

input_path = "E:/LogAnomalyDetector/data/preprocessed_logs.csv"

df = pd.read_csv(input_path)

print("Loaded preprocessed dataset:")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head())


Loaded preprocessed dataset:
Shape: (5, 3)
Columns: ['timestamp', 'message', 'true_label']
             timestamp                          message  true_label
0  2025-07-23 16:11:43      level=info ip=192.168.1.100           0
1  2025-07-23 16:13:22   level=warning ip=192.168.1.101           0
2  2025-07-23 16:15:11               level=error ip=nan           1
3  2025-07-23 16:18:53                level=info ip=nan           0
4  2025-07-23 16:21:08  level=critical ip=192.168.1.105           1


In [6]:
X_text = df["message"].astype(str)
y = df["true_label"].astype(int)

print("\nLabel distribution:")
print(y.value_counts())

print("\nSample messages:")
print(X_text.head())



Label distribution:
true_label
0    3
1    2
Name: count, dtype: int64

Sample messages:
0        level=info ip=192.168.1.100
1     level=warning ip=192.168.1.101
2                 level=error ip=nan
3                  level=info ip=nan
4    level=critical ip=192.168.1.105
Name: message, dtype: object


In [7]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),        # unigrams + bigrams
    max_features=300,         # cap to ~300 features (stable, professional)
    min_df=1,                 # keep all tokens (small dataset)
    stop_words=None,          # do NOT remove stopwords for logs
    token_pattern=r"(?u)\b\w+\b"  # keep numeric tokens like IP parts
)

X = vectorizer.fit_transform(X_text)

print("\nTF-IDF matrix shape:", X.shape)
print("Number of features:", len(vectorizer.get_feature_names_out()))



TF-IDF matrix shape: (5, 28)
Number of features: 28


In [8]:
feature_names = vectorizer.get_feature_names_out()

print("\nFirst 30 feature names:")
print(feature_names[:30])



First 30 feature names:
['1' '1 100' '1 101' '1 105' '100' '101' '105' '168' '168 1' '192'
 '192 168' 'critical' 'critical ip' 'error' 'error ip' 'info' 'info ip'
 'ip' 'ip 192' 'ip nan' 'level' 'level critical' 'level error'
 'level info' 'level warning' 'nan' 'warning' 'warning ip']


In [9]:
import joblib
import os

# Save feature matrix and labels
X_path = "E:/LogAnomalyDetector/data/X_features.npz"
y_path = "E:/LogAnomalyDetector/data/y_labels.csv"
vec_path = "E:/LogAnomalyDetector/models/tfidf_vectorizer.joblib"

os.makedirs("E:/LogAnomalyDetector/models", exist_ok=True)

# Save sparse matrix
from scipy import sparse
sparse.save_npz(X_path, X)

# Save labels
pd.DataFrame({"true_label": y}).to_csv(y_path, index=False)

# Save vectorizer
joblib.dump(vectorizer, vec_path)

print("\nSaved feature matrix to:", X_path)
print("Saved labels to:", y_path)
print("Saved vectorizer to:", vec_path)



Saved feature matrix to: E:/LogAnomalyDetector/data/X_features.npz
Saved labels to: E:/LogAnomalyDetector/data/y_labels.csv
Saved vectorizer to: E:/LogAnomalyDetector/models/tfidf_vectorizer.joblib
